# 🚗 ParkSpot Vision — Kaggle Notebook v5 FINAL FIX

## Root cause of v4 failure

The v4 notebook symlinked the whole `images` directory back to Kaggle Input. Ultralytics resolved that symlink and scanned the original read-only folder, then tried to create:

```text
/kaggle/input/.../train.cache
```

That caused the crash.

## v5 strategy

```text
COCO JSON
→ use only real annotated classes
→ COPY images into writable /kaggle/working
→ generate YOLO labels next to copied images
→ train YOLO11n
→ evaluate
→ export compact outputs
```

Dataset:
```text
ammarnassanalhajali/pklot-dataset
```

Final output:
```text
/kaggle/working/parkspot_outputs.zip
```

In [1]:
from pathlib import Path
import json, random, shutil, zipfile, warnings, time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASET_HANDLE = "ammarnassanalhajali/pklot-dataset"
ROOT = Path("/kaggle/input/datasets/ammarnassanalhajali/pklot-dataset")

WORK = Path("/kaggle/working/parkspot")
PREPARED = WORK / "prepared_yolo"
RUNS = WORK / "runs"
OUTPUTS = WORK / "outputs"
ASSETS = OUTPUTS / "assets"
REPORTS = OUTPUTS / "reports"
MODELS = OUTPUTS / "models"

if WORK.exists():
    shutil.rmtree(WORK)

for p in [PREPARED, RUNS, ASSETS, REPORTS, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

print("Torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))

if not ROOT.exists():
    raise FileNotFoundError(ROOT)

Torch: 2.11.0+cu128
CUDA : True
GPU 0: Tesla T4


## 1. Inspect source dataset

In [2]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_images(root):
    root = Path(root)
    return sorted([
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])

def find_jsons(root):
    return sorted(Path(root).rglob("*.json"))

def choose_coco_json(split_dir):
    candidates = find_jsons(split_dir)
    if not candidates:
        raise FileNotFoundError(f"No JSON in {split_dir}")
    preferred = [p for p in candidates if "annotation" in p.name.lower() or "coco" in p.name.lower()]
    return preferred[0] if preferred else candidates[0]

split_info = {}
coco_paths = {}

for split in ["train", "valid", "test"]:
    imgs = find_images(ROOT / split)
    coco = choose_coco_json(ROOT / split)
    split_info[split] = {"images": imgs}
    coco_paths[split] = coco
    print(f"{split}: {len(imgs)} images | {coco.name}")

train: 8691 images | _annotations.coco.json
valid: 2483 images | _annotations.coco.json
test: 1242 images | _annotations.coco.json


## 2. Keep only classes actually used by annotations

In [3]:
used_names = set()
source_summary = {}

for split, path in coco_paths.items():
    data = json.loads(path.read_text(encoding="utf-8"))

    id_to_name = {
        int(cat["id"]): str(cat["name"])
        for cat in data.get("categories", [])
    }

    counts = Counter()

    for ann in data.get("annotations", []):
        name = id_to_name.get(int(ann["category_id"]))
        if name is not None:
            counts[name] += 1
            used_names.add(name)

    source_summary[split] = {
        "images": len(data.get("images", [])),
        "annotations": len(data.get("annotations", [])),
        "counts": dict(counts),
    }

    print(split, source_summary[split])

class_names = sorted(used_names)
name_to_yolo = {name: i for i, name in enumerate(class_names)}

print("Classes:", class_names)

assert class_names == ["space-empty", "space-occupied"], class_names

train {'images': 8691, 'annotations': 497856, 'counts': {'space-occupied': 231948, 'space-empty': 265908}}
valid {'images': 2483, 'annotations': 143316, 'counts': {'space-occupied': 69687, 'space-empty': 73629}}
test {'images': 1242, 'annotations': 70684, 'counts': {'space-empty': 36584, 'space-occupied': 34100}}
Classes: ['space-empty', 'space-occupied']


## 3. Copy images locally and convert COCO → YOLO

In [4]:
for split in ["train", "valid", "test"]:
    (PREPARED / split / "images").mkdir(parents=True, exist_ok=True)
    (PREPARED / split / "labels").mkdir(parents=True, exist_ok=True)

def convert_split(split):
    data = json.loads(coco_paths[split].read_text(encoding="utf-8"))

    id_to_name = {
        int(cat["id"]): str(cat["name"])
        for cat in data.get("categories", [])
    }

    images_meta = {int(img["id"]): img for img in data.get("images", [])}

    anns_by_image = defaultdict(list)
    for ann in data.get("annotations", []):
        anns_by_image[int(ann["image_id"])].append(ann)

    lookup = {p.name: p for p in split_info[split]["images"]}

    out_images = PREPARED / split / "images"
    out_labels = PREPARED / split / "labels"

    copied = skipped = invalid = boxes = 0

    for image_id, meta in images_meta.items():
        filename = Path(meta["file_name"]).name
        src = lookup.get(filename)

        if src is None:
            matches = list((ROOT / split).rglob(filename))
            src = matches[0] if matches else None

        if src is None or not src.exists():
            skipped += 1
            continue

        shutil.copy2(src, out_images / filename)
        copied += 1

        width = float(meta["width"])
        height = float(meta["height"])
        lines = []

        for ann in anns_by_image.get(image_id, []):
            bbox = ann.get("bbox")
            if not bbox or len(bbox) != 4:
                invalid += 1
                continue

            x, y, w, h = map(float, bbox)
            if width <= 0 or height <= 0 or w <= 0 or h <= 0:
                invalid += 1
                continue

            name = id_to_name.get(int(ann["category_id"]))
            if name not in name_to_yolo:
                continue

            cls = name_to_yolo[name]
            xc = float(np.clip((x + w/2) / width, 0, 1))
            yc = float(np.clip((y + h/2) / height, 0, 1))
            wn = float(np.clip(w / width, 0, 1))
            hn = float(np.clip(h / height, 0, 1))

            if wn <= 0 or hn <= 0:
                invalid += 1
                continue

            lines.append(f"{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}")
            boxes += 1

        (out_labels / f"{Path(filename).stem}.txt").write_text(
            "\n".join(lines), encoding="utf-8"
        )

    return {
        "split": split,
        "images_copied": copied,
        "skipped": skipped,
        "invalid_boxes": invalid,
        "boxes": boxes,
    }

conversion_df = pd.DataFrame([convert_split(s) for s in ["train", "valid", "test"]])
display(conversion_df)
conversion_df.to_csv(REPORTS / "conversion_report.csv", index=False)

,split,images_copied,skipped,invalid_boxes,boxes
0,train,8691,0,0,497856
1,valid,2483,0,0,143316
2,test,1242,0,0,70684


## 4. Validate prepared dataset

In [5]:
def prepared_stats(split):
    imgs = find_images(PREPARED / split / "images")
    labels = sorted((PREPARED / split / "labels").glob("*.txt"))

    counts = Counter()
    total = empty = 0

    for lp in labels:
        lines = [x.strip() for x in lp.read_text(encoding="utf-8").splitlines() if x.strip()]
        if not lines:
            empty += 1
            continue

        for line in lines:
            parts = line.split()
            if len(parts) == 5:
                counts[int(parts[0])] += 1
                total += 1

    return {
        "split": split,
        "images": len(imgs),
        "labels": len(labels),
        "boxes": total,
        "empty_labels": empty,
        "class_counts": dict(counts),
    }

stats = [prepared_stats(s) for s in ["train", "valid", "test"]]

for s in stats:
    print(s)

assert stats[0]["images"] == stats[0]["labels"]
assert stats[0]["boxes"] > 0

{'split': 'train', 'images': 8691, 'labels': 8691, 'boxes': 497856, 'empty_labels': 189, 'class_counts': {1: 231948, 0: 265908}}
{'split': 'valid', 'images': 2483, 'labels': 2483, 'boxes': 143316, 'empty_labels': 59, 'class_counts': {1: 69687, 0: 73629}}
{'split': 'test', 'images': 1242, 'labels': 1242, 'boxes': 70684, 'empty_labels': 26, 'class_counts': {1: 34100, 0: 36584}}


## 5. Create YOLO YAML

In [6]:
import yaml

DATA_YAML = WORK / "parkspot_data.yaml"

cfg = {
    "path": str(PREPARED),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 2,
    "names": class_names,
}

DATA_YAML.write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print(DATA_YAML.read_text())

path: /kaggle/working/parkspot/prepared_yolo
train: train/images
val: valid/images
test: test/images
nc: 2
names:
- space-empty
- space-occupied



## 6. Install / import Ultralytics

In [7]:
try:
    from ultralytics import YOLO
    import ultralytics
except ImportError:
    !pip -q install ultralytics
    from ultralytics import YOLO
    import ultralytics

print("Ultralytics:", ultralytics.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 5.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.173


## 7. Preflight

In [8]:
disk = shutil.disk_usage("/kaggle/working")

prepared_bytes = sum(
    p.stat().st_size
    for p in PREPARED.rglob("*")
    if p.is_file()
)

print("Disk free GB    :", round(disk.free / 1024**3, 2))
print("Prepared size GB:", round(prepared_bytes / 1024**3, 2))
print("Train images    :", stats[0]["images"])
print("Train boxes     :", stats[0]["boxes"])
print("Classes         :", class_names)

Disk free GB    : 18.6
Prepared size GB: 0.84
Train images    : 8691
Train boxes     : 497856
Classes         : ['space-empty', 'space-occupied']


## 8. Train YOLO11n

In [9]:
EPOCHS = 12
IMGSZ = 512
BATCH = 8
PATIENCE = 3
DEVICE = 0 if torch.cuda.is_available() else "cpu"

TRAIN_DIR = RUNS / "parkspot_yolo11n"

if TRAIN_DIR.exists():
    shutil.rmtree(TRAIN_DIR)

model = YOLO("yolo11n.pt")

start = time.time()

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    patience=PATIENCE,
    project=str(RUNS),
    name="parkspot_yolo11n",
    exist_ok=True,
    seed=SEED,
    pretrained=True,
    plots=True,
    cache=False,
    amp=True,
    val=True,
    verbose=True,
)

training_minutes = (time.time() - start) / 60
print(f"Training minutes: {training_minutes:.2f}")

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/parkspot/parkspot_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=12, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=parkspot_yolo11n, nbs=64, nms=None, opset=None,

## 9. Export best model

In [10]:
BEST_PT = TRAIN_DIR / "weights" / "best.pt"
if not BEST_PT.exists():
    raise FileNotFoundError(BEST_PT)

FINAL_PT = MODELS / "parkspot_yolo11n_best.pt"
shutil.copy2(BEST_PT, FINAL_PT)

print("Saved:", FINAL_PT)

Saved: /kaggle/working/parkspot/outputs/models/parkspot_yolo11n_best.pt


## 10. Evaluate

In [11]:
best_model = YOLO(str(BEST_PT))

metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    plots=True,
    project=str(RUNS),
    name="parkspot_test",
    exist_ok=True,
)

summary = {
    "model": "YOLO11n",
    "classes": class_names,
    "epochs_requested": EPOCHS,
    "image_size": IMGSZ,
    "batch_size": BATCH,
    "training_minutes": training_minutes,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
}

print(json.dumps(summary, indent=2))

(REPORTS / "test_metrics.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1649.3±328.4 MB/s, size: 65.6 KB)
val: Scanning /kaggle/working/parkspot/prepared_yolo/test/labels... 1242 images, 26 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1242/1242 1.1Kit/s 1.1s
val: New cache created: /kaggle/working/parkspot/prepared_yolo/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 156/156 9.2it/s 17.0s
                   all       1242      70684      0.997      0.994      0.994       0.93
           space-empty       1041      36584      0.995      0.991      0.993      0.939
        space-occupied        991      34100      0.998      0.996      0.994      0.921
Speed: 0.3ms preprocess, 4.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /kaggle/working/parksp

323

## 11. Sample predictions

In [12]:
test_images = find_images(PREPARED / "test" / "images")
samples = random.sample(test_images, min(8, len(test_images)))

results = best_model.predict(
    source=[str(p) for p in samples],
    imgsz=IMGSZ,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(OUTPUTS),
    name="prediction_samples",
    exist_ok=True,
    verbose=False,
)

rows = []

for result in results:
    counts = {"space-empty": 0, "space-occupied": 0}

    if result.boxes is not None and len(result.boxes):
        ids = result.boxes.cls.detach().cpu().numpy().astype(int)

        for cid in ids:
            counts[class_names[cid]] += 1

    rows.append({
        "image": Path(result.path).name,
        "estimated_vacant": counts["space-empty"],
        "estimated_occupied": counts["space-occupied"],
        "total_detected_slots": sum(counts.values()),
    })

prediction_df = pd.DataFrame(rows)
display(prediction_df)

prediction_df.to_csv(
    REPORTS / "sample_detection_counts.csv",
    index=False,
)

Results saved to /kaggle/working/parkspot/outputs/prediction_samples


,image,estimated_vacant,estimated_occupied,total_detected_slots
0,2013-01-29_20_16_22_jpg.rf.a15f18a0b3e3732edb3...,4,26,30
1,2013-04-11_11_45_06_jpg.rf.6212987299570f080e2...,0,41,41
2,2012-10-26_17_39_59_jpg.rf.dffd35b85f996b9a2a7...,48,52,100
3,2012-12-18_11_35_07_jpg.rf.1e3e6269ce14081b14e...,1,27,28
4,2013-03-06_16_10_12_jpg.rf.f9d94a19e094a4e764d...,0,40,40
5,2013-03-06_15_20_11_jpg.rf.549cf293aceef8d6917...,3,38,41
6,2012-12-20_15_35_10_jpg.rf.cae0442adee78d2529b...,0,28,28
7,2013-04-14_07_35_01_jpg.rf.7a249550b6a2eb0e1e2...,40,0,40


## 12. Copy useful plots

In [13]:
for filename in [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
]:
    src = TRAIN_DIR / filename
    if src.exists():
        shutil.copy2(src, ASSETS / filename)

## 13. Save reports

In [14]:
grounding = {
    "dataset_handle": DATASET_HANDLE,
    "source_format": "COCO",
    "training_format": "YOLO",
    "classes": class_names,
    "source_summary": source_summary,
    "prepared_stats": stats,
    "limitations": [
        "Uses the Kaggle package train/valid/test split.",
        "Original camera/day grouping may not be preserved.",
        "Predictions are frame-level estimates.",
        "Live CCTV and temporal smoothing are not implemented."
    ]
}

final_summary = {
    "project": "ParkSpot Vision",
    "progress": "3%",
    "model": "YOLO11n",
    "metrics": summary,
    "grounding": grounding,
}

(REPORTS / "grounding_summary.json").write_text(
    json.dumps(grounding, indent=2), encoding="utf-8"
)

(REPORTS / "final_summary.json").write_text(
    json.dumps(final_summary, indent=2), encoding="utf-8"
)

2166

## 14. Clean temporary files

In [15]:
if PREPARED.exists():
    shutil.rmtree(PREPARED)

if RUNS.exists():
    shutil.rmtree(RUNS)

print("Temporary dataset and runs removed.")

Temporary dataset and runs removed.


## 15. Package outputs

In [16]:
ZIP_PATH = Path("/kaggle/working/parkspot_outputs.zip")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in OUTPUTS.rglob("*"):
        if fp.is_file():
            zf.write(fp, fp.relative_to(OUTPUTS))

print("DONE")
print("ZIP:", ZIP_PATH)
print("ZIP MB:", round(ZIP_PATH.stat().st_size / 1024**2, 2))

DONE
ZIP: /kaggle/working/parkspot_outputs.zip
ZIP MB: 6.72
